# The OTF Acting on a Real Object


> Part of *PHYS 0241: Biomedical Imaging* (Prof. Michael E. Durst, Middlebury College). The [textbook page for this notebook](https://durstlab.middcreate.net/teaching/protected/textbook-biomedicalimaging/python/ch05/ch05-19.html) shows the rendered version together with the pre-rendered animations (course password required).
>
> **Start here:** no setup is needed for this notebook. In Google Colab, choose **File > Save a copy in Drive** to keep your changes.
>
> *Provided for academic use in PHYS 0241; please do not repost or redistribute ([terms of use](https://github.com/durstme/textbook-biomedicalimaging-notebooks/blob/main/LICENSE.md)).*

The chapter has just shown that convolving in space is multiplying in frequency:

$$G(k) = F(k) \cdot \textrm{OTF}(k)$$

Every worked example so far has fed that equation a tidy shape, a rect or a cosine,
chosen because the algebra comes out clean. This notebook feeds it something closer to
an actual specimen: a bright region with texture on it, structure at several scales at
once, and a little noise.

The point is what the equation then does. The OTF is **zero beyond a cutoff**, so
multiplying by it does not merely soften the object's fine detail. It **deletes** it.
Anything the object contains above the cutoff frequency contributes exactly nothing to
the image, and no amount of processing afterwards can bring it back, because it is not
there to be recovered.

To make that visible rather than assert it, the object below carries three ripples of
**identical amplitude** at three spatial frequencies: one comfortably inside the
cutoff, one near it, and one outside. Watch what happens to each.


## Import libraries

NumPy and Matplotlib only. `ConnectionPatch` draws the arrows that turn six separate
axes into one flow diagram, exactly as in `ch05-15` and `ch05-16`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import ConnectionPatch


In [ ]:
%matplotlib inline
# ==========================================
# 1. Configuration & Styling
# ==========================================
plot_params = {
    'font.size': 14,
    'font.family': 'sans-serif',
    'mathtext.fontset': 'stix',
    'axes.linewidth': 1.0,
    'lines.linewidth': 2.5,
}
plt.rcParams.update(plot_params)


def apply_custom_formatting(ax):
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.minorticks_on()
    ax.tick_params(which='both', direction="in", width=1)
    ax.tick_params(which='major', length=7)
    ax.tick_params(which='minor', length=3.5)
    ax.xaxis.set_ticks_position('both')
    ax.yaxis.set_ticks_position('both')


## Define parameters

**A 1D slit, not a circular aperture.** This is the same simplification `ch05-13`
makes, and it is what keeps every panel a shape you already recognize: the pupil is a
rect, the incoherent PSF is a sinc squared, and the OTF is the rect autocorrelated with
itself, which is a **triangle**. A circular aperture would replace the triangle with the
Bessel expression from `ch05-06` and change nothing about the argument.

**The instrument is the one from `ch05-17` and `ch05-18`**: a 25 mm aperture at 100 mm
focal length in green light, so $\lambda f / D = 2.2~\mu$m. That single number fixes
both cutoffs,

$$k_\textrm{coh} = \frac{\pi D}{\lambda f}, \qquad
k_\textrm{inc} = 2 k_\textrm{coh} = \frac{2\pi D}{\lambda f}$$

and the incoherent cutoff corresponds to a period of exactly $\lambda f / D = 2.2~\mu$m.
Structure finer than that does not survive the lens.

**The three test ripples** have periods of 8.0, 3.2 and 1.6 $\mu$m, which is
$0.27\,k_\textrm{inc}$, $0.69\,k_\textrm{inc}$ and $1.37\,k_\textrm{inc}$. They are
given the **same amplitude** in the object so that any difference in the image is the
OTF's doing and nothing else.

**The window is chosen so the ripples fit it a whole number of times.** $L = 80~\mu$m
holds exactly 10, 25 and 50 cycles of the three periods. That is not fussiness: a
Fourier transform assumes the signal repeats, so a ripple that does not close on itself
at the edges smears across neighboring frequencies and muddies the very peaks this
figure is about.

**The noise is seeded** so the "real object" texture is the same every run.


In [ ]:
# ==========================================
# 2. Data Generation
# ==========================================
# All lengths in micrometers. Same instrument as ch05-17 and ch05-18.
lam1 = 0.55        # wavelength, green light
f_lens1 = 1.0e5    # focal length, 100 mm
diam1 = 2.5e4      # slit width, 25 mm

scale1 = lam1 * f_lens1 / diam1          # lambda f / D = 2.2 um
k_coh1 = np.pi / scale1                  # coherent cutoff
k_inc1 = 2 * k_coh1                      # incoherent cutoff, = 2 pi / scale


def unnormalized_sinc(u):
    """sinc(u) = sin(u)/u, this text's convention.

    np.sinc is the NORMALIZED sin(pi z)/(pi z), so divide the argument by pi.
    """
    return np.sinc(u / np.pi)


# --- The grid. L holds a whole number of cycles of every test ripple ---
L1 = 80.0
N1 = 16384
y1 = np.linspace(-L1 / 2, L1 / 2, N1, endpoint=False)
dy1 = y1[1] - y1[0]
k1 = 2 * np.pi * np.fft.fftshift(np.fft.fftfreq(N1, d=dy1))

# --- The object: a bright region with texture at three scales ---
periods1 = np.array([8.0, 3.2, 1.6])     # 10, 25 and 50 cycles across L
k_ripple1 = 2 * np.pi / periods1
ripple_amp1 = 0.35                       # the SAME for all three

envelope1 = 0.15 + 0.85 * np.exp(-(y1 / 11.0) ** 2)
texture1 = sum(ripple_amp1 * np.cos(kr * y1) for kr in k_ripple1)

rng1 = np.random.default_rng(4)          # seeded, so the figure is reproducible
noise1 = 0.03 * rng1.standard_normal(N1)

obj1 = envelope1 * (1 + texture1) + noise1

# --- The system: incoherent PSF and the OTF that goes with it ---
psf1 = unnormalized_sinc(np.pi * y1 / scale1) ** 2       # |h|^2, the magnitude squared
otf1 = np.clip(1 - np.abs(k1) / k_inc1, 0, None)         # triangle, = rect (*) rect

# --- The image: multiply the spectra, which IS the convolution ---
F_obj1 = np.fft.fftshift(np.fft.fft(np.fft.ifftshift(obj1))) * dy1
F_img1 = F_obj1 * otf1
img1 = np.real(np.fft.fftshift(np.fft.ifft(np.fft.ifftshift(F_img1)))) / dy1

# OTF value at each test ripple, for the annotations
otf_at1 = np.clip(1 - k_ripple1 / k_inc1, 0, None)


## Plot the grid

The same 2x3 flow diagram as `ch05-15` and `ch05-16`, now with an object worth blurring.

**The top row is the space domain.** The object on the left has visible fine texture.
The image on the right does not: it keeps the broad shape and the coarse ripple, and
the finest ripple has vanished completely. Nothing was thrown away by hand; the only
operation applied was a convolution with the PSF in the middle panel.

**The bottom row is the frequency domain, and it explains the top row.** The object's
spectrum has three marked peaks, one per test ripple, all the same height because the
ripples had the same amplitude. The OTF is the triangle beside it. Multiplying gives the
image spectrum, and the dashed line at $k_\textrm{inc}$ is where the story ends: the
shaded region beyond it is **empty**, in the image spectrum and in the image.

Read the three peaks across the bottom row and the whole chapter is in one picture:

| ripple | position | OTF there | fate |
|---|---|---|---|
| 8.0 $\mu$m | $0.27\,k_\textrm{inc}$ | 0.73 | survives, slightly reduced |
| 3.2 $\mu$m | $0.69\,k_\textrm{inc}$ | 0.31 | badly attenuated but present |
| 1.6 $\mu$m | $1.37\,k_\textrm{inc}$ | 0 | gone |

**The middle ripple is the one to dwell on.** It is well inside the cutoff, so a
resolution figure of merit would call it resolved, and yet it arrives at less than a
third of its true contrast. A cutoff tells you where an instrument stops. The OTF tells
you how it was already doing before it got there.

**The PSF panel is drawn on an expanded horizontal scale**, marked on the panel. It has
to be: the kernel is only a couple of micrometers wide against an object tens of
micrometers across, which is itself the reason the image keeps its broad shape and
loses only the fine detail.


In [ ]:
# ==========================================
# 3. Plotting the Grid
# ==========================================
fig1, axes1 = plt.subplots(2, 3, figsize=(17, 9))
plt.subplots_adjust(wspace=0.46, hspace=0.78, left=0.115, right=0.975,
                    top=0.90, bottom=0.09)

xlim1 = 17.0        # object and image
xlim_psf1 = 6.0     # the kernel is far narrower, so this panel is expanded
klim1 = 1.75 * k_inc1

# --- Top Row: Space Domain ---
ax_f = axes1[0, 0]
ax_f.plot(y1, obj1, 'r', linewidth=1.6)
ax_f.set_title(r"$I_\mathrm{obj}(y)$,  a real object", fontsize=16, color='red')
ax_f.set_xlim(-xlim1, xlim1)
ax_f.set_ylim(-0.15, 2.3)
ax_f.set_xlabel(r"$y$  ($\mu$m)")
apply_custom_formatting(ax_f)

ax_h = axes1[0, 1]
ax_h.plot(y1, psf1, 'b')
ax_h.fill_between(y1, 0, psf1, color='blue', alpha=0.12)
ax_h.set_title(r"$\left| h(y) \right|^2$,  the incoherent PSF", fontsize=16, color='blue')
ax_h.set_xlim(-xlim_psf1, xlim_psf1)
ax_h.set_ylim(-0.08, 1.20)
ax_h.set_xlabel(r"$y$  ($\mu$m)")
ax_h.text(0.97, 0.90, "expanded scale", transform=ax_h.transAxes, fontsize=11,
          color='blue', ha='right', va='top', style='italic')
apply_custom_formatting(ax_h)

ax_g = axes1[0, 2]
ax_g.plot(y1, img1, 'g', linewidth=1.8)
ax_g.set_title(r"$I_\mathrm{image}(Y) = I_\mathrm{obj} \otimes \left| h \right|^2$", fontsize=16,
               color='green')
ax_g.set_xlim(-xlim1, xlim1)
ax_g.set_ylim(-0.15, 2.3)
ax_g.set_xlabel(r"$Y$  ($\mu$m)")
apply_custom_formatting(ax_g)

# --- Bottom Row: Frequency Domain ---
mag_obj1 = np.abs(F_obj1)
mag_img1 = np.abs(F_img1)
# Scale the axis to the ripple peaks, not to the much larger DC lump
peak_ref1 = mag_obj1[np.argmin(np.abs(k1 - k_ripple1[0]))] * 2.6

ax_F = axes1[1, 0]
ax_F.plot(k1, mag_obj1, 'darkred', linewidth=1.4)
ax_F.set_title(r"$\mathscr{F}\{I_\mathrm{obj}\}$", fontsize=16, color='darkred')
ax_F.set_xlim(-klim1, klim1)
ax_F.set_ylim(0, peak_ref1)
ax_F.set_xlabel(r"$k$  (rad/$\mu$m)")
apply_custom_formatting(ax_F)

ax_H = axes1[1, 1]
ax_H.plot(k1, otf1, 'darkblue')
ax_H.fill_between(k1, 0, otf1, color='blue', alpha=0.12)
ax_H.set_title(r"$\mathrm{OTF}(k)$", fontsize=16, color='darkblue')
ax_H.set_xlim(-klim1, klim1)
ax_H.set_ylim(0, 1.20)
ax_H.set_xlabel(r"$k$  (rad/$\mu$m)")
apply_custom_formatting(ax_H)

ax_G = axes1[1, 2]
ax_G.plot(k1, mag_img1, 'darkgreen', linewidth=1.4)
ax_G.set_title(r"$\mathscr{F}\{I_\mathrm{image}\} = \mathscr{F}\{I_\mathrm{obj}\}"
               r"\cdot \mathrm{OTF}$", fontsize=16, color='darkgreen')
ax_G.set_xlim(-klim1, klim1)
ax_G.set_ylim(0, peak_ref1)
ax_G.set_xlabel(r"$k$  (rad/$\mu$m)")
apply_custom_formatting(ax_G)

# --- The cutoff, and what lies past it ---
for ax_b1 in (ax_F, ax_H, ax_G):
    for sgn1 in (-1, 1):
        ax_b1.axvline(sgn1 * k_inc1, color='black', linestyle='--', linewidth=1.4)
    top1 = ax_b1.get_ylim()[1]
    ax_b1.fill_between([k_inc1, klim1], 0, top1, color='gray', alpha=0.18)
    ax_b1.fill_between([-klim1, -k_inc1], 0, top1, color='gray', alpha=0.18)
ax_H.text(k_inc1 * 1.05, 0.72, r"$k_\mathrm{inc}$", fontsize=15, ha='left')
ax_G.text(k_inc1 * 1.35, peak_ref1 * 0.55, "nothing\nout here", fontsize=12,
          color='dimgray', ha='center', va='center')

# --- Label the three test ripples in both spectra ---
# The whole argument rests on the three going IN at the same height, so say so
# once in the object panel rather than drawing three unexplained arrows.
h_equal1 = mag_obj1[np.argmin(np.abs(k1 - k_ripple1[0]))]
ax_F.plot([k_ripple1[0] * 0.55, k_ripple1[-1] * 1.18], [h_equal1, h_equal1],
          color='dimgray', linestyle=':', linewidth=1.6)
ax_F.text(k_ripple1[1], h_equal1 * 1.10, "all three the same amplitude",
          fontsize=11, color='dimgray', ha='center', va='bottom')

tags1 = ["survives", "attenuated", "erased"]
for kr1, otf_v1, tag1 in zip(k_ripple1, otf_at1, tags1):
    i_k1 = np.argmin(np.abs(k1 - kr1))
    for ax_b1, mag1 in ((ax_F, mag_obj1), (ax_G, mag_img1)):
        ax_b1.plot(kr1, mag1[i_k1], 'o', color='black', markersize=6, zorder=6)
    # Tag sits just above its own peak, so it cannot reach the panel title
    ax_G.text(kr1, mag_img1[i_k1] + peak_ref1 * 0.04, tag1, fontsize=11,
              rotation=90, color='darkgreen', ha='center', va='bottom')
    ax_H.annotate("%.2f" % otf_v1, xy=(kr1, otf_v1), xytext=(kr1, otf_v1 + 0.12),
                  fontsize=12, color='darkblue', ha='center',
                  arrowprops=dict(arrowstyle="->", color='darkblue', linewidth=1.2))

# ==========================================
# 4. Adding Flowchart Annotations
# ==========================================
# Positions are read back from the axes so the glyphs land in the gutters
# between columns rather than on top of a panel.
fig1.canvas.draw()


def draw_arrow(ax_from, ax_to, text=""):
    """Vertical arrow between a column's two panels, as in ch05-10.

    Axes-fraction coordinates with axesA/axesB, added through add_artist.
    Passing figure-fraction coords together with axesA is not valid and
    raises AttributeError on transFigure.
    """
    con = ConnectionPatch(xyA=(0.5, -0.32), xyB=(0.5, 1.14),
                          coordsA="axes fraction", coordsB="axes fraction",
                          axesA=ax_from, axesB=ax_to,
                          arrowstyle="-|>", shrinkB=5, linewidth=2.5,
                          color="black")
    ax_from.add_artist(con)
    if text:
        ax_to.text(0.56, 1.20, text, transform=ax_to.transAxes,
                   ha='left', va='bottom', fontsize=15)


draw_arrow(ax_f, ax_F, text="F.T.")
draw_arrow(ax_h, ax_H, text="F.T.")
draw_arrow(ax_g, ax_G, text="F.T.")


def gutter(ax_left, ax_right):
    """Center of the CLEAR gap between two side-by-side panels.

    The right panel's y-tick labels stick out to the left of its axes box, so
    centering between the boxes alone drops the glyph on top of them. Measuring
    where those labels actually begin gives a gap that is genuinely empty, and
    removes the need for a hand-tuned nudge.
    """
    fig = ax_left.figure
    renderer = fig.canvas.get_renderer()
    inv = fig.transFigure.inverted()
    right_x0 = ax_right.get_position().x0
    for lab in ax_right.get_yticklabels():
        if lab.get_visible() and lab.get_text():
            right_x0 = min(
                right_x0,
                lab.get_window_extent(renderer).transformed(inv).x0)
    return (ax_left.get_position().x1 + right_x0) / 2


def row_mid(ax):
    p = ax.get_position()
    return (p.y0 + p.y1) / 2


gx1, gx2 = gutter(ax_f, ax_h), gutter(ax_h, ax_g)
y_top, y_bot = row_mid(ax_f), row_mid(ax_F)

# One x for both the operator word and its symbol, so they stay aligned.
# gutter() already accounts for the tick labels, so no nudge is wanted here:
# the old one pushed the word onto the left panel's frame.
op_x = gx1

fig1.text(op_x, y_top + 0.055, "convolve", fontsize=12, ha='center', va='center')
fig1.text(op_x, y_top, r"$\otimes$", fontsize=32, ha='center', va='center')
fig1.text(gx2, y_top, r"$=$", fontsize=32, ha='center', va='center')

fig1.text(op_x, y_bot + 0.055, "multiply", fontsize=12, ha='center', va='center')
fig1.text(op_x, y_bot, r"$\times$", fontsize=32, ha='center', va='center')
fig1.text(gx2, y_bot, r"$=$", fontsize=32, ha='center', va='center')

# Row labels, to the left of the first column
fig1.text(0.035, y_top, "Space\nDomain", fontsize=17, weight='bold',
          rotation=90, va='center', ha='center')
fig1.text(0.035, y_bot, "Frequency\nDomain", fontsize=17, weight='bold',
          rotation=90, va='center', ha='center')

plt.show()


### Save plot

Commented out on purpose. Uncomment it to regenerate the figure used in the chapter;
leaving it live would rewrite that file every time the notebook is run.


In [ ]:
# fig1.savefig("../figures/ch05/ch05-19-otf-real-object.jpg", dpi=200, bbox_inches='tight')
